In [1]:
# ============================================
# 01_data_preparation.ipynb (COMPLETE FIXED VERSION)
# Purpose: Scrape Pakistan history from Wikipedia and create chunks
# ============================================

# Cell 1: Install and import libraries
import wikipedia as wiki
import time
import json
from typing import List, Dict
import nltk
import pandas as pd
import random
import os

# Download NLTK data
try:
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)
    print("✅ NLTK data downloaded")
except:
    print("⚠ NLTK download issue, but continuing...")

print("Libraries imported successfully!")

✅ NLTK data downloaded
Libraries imported successfully!


In [2]:
# ============================================
# Cell 2: Define Wikipedia pages to scrape (100+ topics)
# ============================================

# Complete list of Pakistan history topics - expanded to 120+ pages
pakistan_topics = [
    # Ancient History
    "Indus Valley Civilisation",
    "Mehrgarh",
    "Vedic period",
    "Achaemenid Empire",
    "Gandhara",
    "Maurya Empire",
    "Indo-Greek Kingdom",
    "Kushan Empire",
    "Gupta Empire",
    "Sassanian Empire",
    "Rai dynasty",
    "Brahman dynasty of Sindh",
    "Umayyad Caliphate",
    "Abbasid Caliphate",
    "Habbari dynasty",
    
    # Medieval History
    "Ghaznavids",
    "Ghurid dynasty",
    "Delhi Sultanate",
    "Mamluk dynasty (Delhi)",
    "Khalji dynasty",
    "Tughlaq dynasty",
    "Sayyid dynasty",
    "Lodi dynasty",
    "Mughal Empire",
    "Babur",
    "Humayun",
    "Akbar",
    "Jahangir",
    "Shah Jahan",
    "Aurangzeb",
    "Durrani Empire",
    "Sikh Empire",
    "Ranjit Singh",
    "Battle of Plassey",
    "Battle of Panipat (1761)",
    
    # Colonial Period
    "British East India Company",
    "Company rule in India",
    "British Raj",
    "Indian Rebellion of 1857",
    "British Indian Army",
    "Syed Ahmad Khan",
    "Aligarh Movement",
    "Indian National Congress",
    "All-India Muslim League",
    "Lucknow Pact",
    "Khilafat Movement",
    "Simon Commission",
    "Nehru Report",
    "Jinnah's Fourteen Points",
    "Allahabad Address",
    "Pakistan Movement",
    "Lahore Resolution",
    "Cripps Mission",
    "Indian Independence Act 1947",
    "Partition of India",
    
    # Independence and Early Years
    "Muhammad Ali Jinnah",
    "Allama Iqbal",
    "Fatima Jinnah",
    "Liaquat Ali Khan",
    "Objective Resolution",
    "Constitution of Pakistan of 1956",
    "Indo-Pakistani War of 1947",
    "Indo-Pakistani War of 1965",
    "Tashkent Declaration",
    "Ayub Khan (President of Pakistan)",
    "Yahya Khan",
    "1970 Pakistani general election",
    "Bangladesh Liberation War",
    "Indo-Pakistani War of 1971",
    "Simla Agreement",
    
    # Post-1971 Pakistan
    "Zulfikar Ali Bhutto",
    "Muhammad Zia-ul-Haq",
    "Islamization in Pakistan",
    "Pakistan and its Nuclear Deterrent Program",
    "Kargil War",
    "Pervez Musharraf",
    "War in North-West Pakistan",
    "Benazir Bhutto",
    "Nawaz Sharif",
    "1999 Pakistani coup d'état",
    "2005 Kashmir earthquake",
    "Pakistan Armed Forces",
    "Siachen conflict",
    "Inter-Services Intelligence",
    
    # Regional Histories
    "History of Sindh",
    "History of Punjab, Pakistan",
    "History of Balochistan",
    "History of Khyber Pakhtunkhwa",
    "History of Gilgit-Baltistan",
    "History of Karachi",
    "History of Lahore",
    "History of Multan",
    "History of Peshawar",
    
    # Additional topics to reach 100+
    "Muhammad bin Qasim",
    "Mahmud of Ghazni",
    "Qutb ud-Din Aibak",
    "Razia Sultana",
    "Sher Shah Suri",
    "Hemu",
    "Chaudhry Rehmat Ali",
    "Huseyn Shaheed Suhrawardy",
    "Khwaja Nazimuddin",
    "Iskander Mirza",
    "Muhammad Khan Junejo",
    "Ghulam Ishaq Khan",
    "Rafiq Tarar",
    "Asif Ali Zardari",
]

# Remove duplicates
pakistan_topics = list(dict.fromkeys(pakistan_topics))
print(f"✅ Total unique topics to scrape: {len(pakistan_topics)}")

✅ Total unique topics to scrape: 107


In [3]:
# ============================================
# Cell 3: Function to scrape Wikipedia with error handling
# ============================================

def scrape_wikipedia_page(page_title: str) -> Dict:
    """
    Scrape a Wikipedia page with error handling
    """
    try:
        # Try to get the exact page
        page = wiki.page(page_title, auto_suggest=False)
        
        return {
            'title': page_title,
            'content': page.content,
            'summary': page.summary,
            'url': page.url,
            'length': len(page.content),
            'success': True
        }
        
    except wiki.exceptions.DisambiguationError as e:
        # If disambiguation, try first option
        try:
            if e.options:
                page = wiki.page(e.options[0], auto_suggest=False)
                return {
                    'title': page_title,
                    'content': page.content,
                    'summary': page.summary,
                    'url': page.url,
                    'length': len(page.content),
                    'success': True
                }
        except:
            pass
        return None
        
    except wiki.exceptions.PageError:
        # Try with "History of " prefix
        try:
            alt_title = f"History of {page_title}"
            page = wiki.page(alt_title, auto_suggest=False)
            return {
                'title': alt_title,
                'content': page.content,
                'summary': page.summary,
                'url': page.url,
                'length': len(page.content),
                'success': True
            }
        except:
            pass
        return None
        
    except Exception as e:
        print(f"    Error: {str(e)[:50]}")
        return None

# Test the function
print("Testing scraper...")
test = scrape_wikipedia_page("Pakistan Movement")
if test:
    print(f"✅ Scraper works! Got {test['length']} characters")
else:
    print("❌ Scraper test failed - check your internet connection")

Testing scraper...
✅ Scraper works! Got 47721 characters


In [4]:
# ============================================
# Cell 4: Scrape all pages
# ============================================

all_documents = []
failed_pages = []

print(f"\n📥 Starting to scrape {len(pakistan_topics)} Wikipedia pages...")
print("="*60)

for i, topic in enumerate(pakistan_topics, 1):
    print(f"[{i}/{len(pakistan_topics)}] {topic}...", end=" ")
    
    doc = scrape_wikipedia_page(topic)
    
    if doc:
        all_documents.append(doc)
        print(f"✅ ({doc['length']:,} chars)")
    else:
        failed_pages.append(topic)
        print(f"❌ Failed")
    
    # Rate limiting - be nice to Wikipedia
    time.sleep(0.5)
    
    # Save checkpoint every 20 pages
    if i % 20 == 0:
        with open('checkpoint_docs.json', 'w') as f:
            json.dump(all_documents, f, indent=2)
        print(f"  💾 Checkpoint saved ({len(all_documents)} docs so far)")

print("\n" + "="*60)
print("SCRAPING COMPLETE!")
print("="*60)
print(f"✅ Successful: {len(all_documents)} pages")
print(f"❌ Failed: {len(failed_pages)} pages")

# If we need more pages, try to get from failed list
if len(all_documents) < 80:
    print(f"\n⚠ Only {len(all_documents)} pages. Attempting to salvage from failed...")
    for failed in failed_pages[:20]:
        # Try with different variations
        variations = [failed, f"Pakistan {failed}", f"{failed} Pakistan"]
        for var in variations:
            doc = scrape_wikipedia_page(var)
            if doc:
                all_documents.append(doc)
                print(f"  ✅ Salvaged: {var}")
                break
            time.sleep(0.3)

print(f"\n📊 Final document count: {len(all_documents)}")

# Save final documents
with open('pakistan_history_raw.json', 'w') as f:
    json.dump(all_documents, f, indent=2)

print(f"\n✅ Saved {len(all_documents)} documents to 'pakistan_history_raw.json'")


📥 Starting to scrape 107 Wikipedia pages...
[1/107] Indus Valley Civilisation... ✅ (65,371 chars)
[2/107] Mehrgarh... ✅ (18,140 chars)
[3/107] Vedic period... ✅ (36,402 chars)
[4/107] Achaemenid Empire... ✅ (78,494 chars)
[5/107] Gandhara... ✅ (42,228 chars)
[6/107] Maurya Empire... ✅ (48,289 chars)
[7/107] Indo-Greek Kingdom... ✅ (86,929 chars)
[8/107] Kushan Empire... ✅ (39,809 chars)
[9/107] Gupta Empire... ✅ (32,082 chars)
[10/107] Sassanian Empire... ✅ (106,430 chars)
[11/107] Rai dynasty... ✅ (9,615 chars)
[12/107] Brahman dynasty of Sindh... ✅ (2,494 chars)
[13/107] Umayyad Caliphate... ✅ (63,606 chars)
[14/107] Abbasid Caliphate... ✅ (74,940 chars)
[15/107] Habbari dynasty... ✅ (8,182 chars)
[16/107] Ghaznavids... ✅ (24,854 chars)
[17/107] Ghurid dynasty... ✅ (28,421 chars)
[18/107] Delhi Sultanate... ✅ (48,754 chars)
[19/107] Mamluk dynasty (Delhi)... ✅ (10,580 chars)
[20/107] Khalji dynasty... ✅ (16,827 chars)
  💾 Checkpoint saved (20 docs so far)
[21/107] Tughlaq dynasty...

In [5]:
# ============================================
# Cell 5: Statistics
# ============================================

print("\n" + "="*60)
print("CORPUS STATISTICS")
print("="*60)

total_chars = sum(d['length'] for d in all_documents)
avg_chars = total_chars / len(all_documents) if all_documents else 0

print(f"📊 Total pages: {len(all_documents)}")
print(f"📊 Total characters: {total_chars:,}")
print(f"📊 Average length: {avg_chars:,.0f} chars")


CORPUS STATISTICS
📊 Total pages: 107
📊 Total characters: 4,220,350
📊 Average length: 39,443 chars


In [7]:
# ============================================
# Cell 6: Create CHUNKS (OPTIMIZED FAST VERSION)
# ============================================

from typing import List, Dict

print("\n" + "="*60)
print("CREATING CHUNKS")
print("="*60)

# Import LangChain's text splitter (much faster and optimized)
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_fixed_size_fast(text: str, chunk_size: int = 500, overlap: int = 50) -> List[Dict]:
    """Strategy 1: Fast fixed-size chunking using LangChain"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        length_function=len,
        separators=[" ", ",", "\n"],  # Simple separators for speed
    )
    
    chunks = text_splitter.split_text(text)
    
    return [{
        'chunk_id': i,
        'text': chunk,
        'chunk_size': len(chunk)
    } for i, chunk in enumerate(chunks) if len(chunk.strip()) > 50]

def chunk_recursive_fast(text: str, chunk_size: int = 500, overlap: int = 50) -> List[Dict]:
    """Strategy 2: Fast recursive chunking using LangChain"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        length_function=len,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    
    chunks = text_splitter.split_text(text)
    
    return [{
        'chunk_id': i,
        'text': chunk,
        'chunk_size': len(chunk)
    } for i, chunk in enumerate(chunks) if len(chunk.strip()) > 50]

# Quick test on a small sample
if all_documents:
    print("📝 Testing chunking strategies on sample text (this should be fast)...")
    
    # Use a smaller sample for testing (500 chars instead of 2000)
    sample_text = all_documents[0]['content'][:500]
    
    fixed_sample = chunk_fixed_size_fast(sample_text)
    print(f"  ✅ Fixed-size chunking: {len(fixed_sample)} chunks (took < 1 second)")
    
    recursive_sample = chunk_recursive_fast(sample_text)
    print(f"  ✅ Recursive chunking: {len(recursive_sample)} chunks (took < 1 second)")
    
    if fixed_sample:
        print(f"\n  Sample fixed chunk: {fixed_sample[0]['text'][:150]}...")
    if recursive_sample:
        print(f"  Sample recursive chunk: {recursive_sample[0]['text'][:150]}...")

# Now chunk ALL documents
print("\n📦 Chunking all documents (this will take 30-60 seconds)...")
print("   Processing", len(all_documents), "documents...")

all_chunks_fixed = []
all_chunks_recursive = []

# Process with progress indicator
from tqdm import tqdm

for idx, doc in enumerate(tqdm(all_documents, desc="Chunking documents")):
    # Fixed-size chunking
    fixed_chunks = chunk_fixed_size_fast(doc['content'])
    for chunk in fixed_chunks:
        all_chunks_fixed.append({
            'source_title': doc['title'],
            'source_url': doc.get('url', ''),
            'chunk_id': chunk['chunk_id'],
            'text': chunk['text'],
            'chunking_strategy': 'fixed',
            'chunk_size': chunk['chunk_size']
        })
    
    # Recursive chunking
    recursive_chunks = chunk_recursive_fast(doc['content'])
    for chunk in recursive_chunks:
        all_chunks_recursive.append({
            'source_title': doc['title'],
            'source_url': doc.get('url', ''),
            'chunk_id': chunk['chunk_id'],
            'text': chunk['text'],
            'chunking_strategy': 'recursive',
            'chunk_size': chunk['chunk_size']
        })
    
    # Show progress every 10 documents
    if (idx + 1) % 10 == 0:
        print(f"   Processed {idx + 1}/{len(all_documents)} docs - Fixed: {len(all_chunks_fixed)} chunks, Recursive: {len(all_chunks_recursive)} chunks")

print(f"\n✅ Fixed-size chunks created: {len(all_chunks_fixed)}")
print(f"✅ Recursive chunks created: {len(all_chunks_recursive)}")


CREATING CHUNKS
📝 Testing chunking strategies on sample text (this should be fast)...
  ✅ Fixed-size chunking: 1 chunks (took < 1 second)
  ✅ Recursive chunking: 1 chunks (took < 1 second)

  Sample fixed chunk: The Indus Valley Civilisation (IVC), also known as the Indus Civilisation or the Harappan Civilisation, was a Bronze Age civilisation in the northwest...
  Sample recursive chunk: The Indus Valley Civilisation (IVC), also known as the Indus Civilisation or the Harappan Civilisation, was a Bronze Age civilisation in the northwest...

📦 Chunking all documents (this will take 30-60 seconds)...
   Processing 107 documents...


   Processed 10/107 docs - Fixed: 1236 chunks, Recursive: 1621 chunks
   Processed 20/107 docs - Fixed: 1883 chunks, Recursive: 2441 chunks
   Processed 30/107 docs - Fixed: 2725 chunks, Recursive: 3543 chunks
   Processed 40/107 docs - Fixed: 3950 chunks, Recursive: 5099 chunks
   Processed 50/107 docs - Fixed: 4459 chunks, Recursive: 5744 chunks


   Processed 60/107 docs - Fixed: 5291 chunks, Recursive: 6822 chunks
   Processed 70/107 docs - Fixed: 5999 chunks, Recursive: 7759 chunks


Chunking documents: 100%|██████████| 107/107 [00:00<00:00, 244.93it/s]

   Processed 80/107 docs - Fixed: 7644 chunks, Recursive: 9900 chunks
   Processed 90/107 docs - Fixed: 8461 chunks, Recursive: 10974 chunks
   Processed 100/107 docs - Fixed: 9060 chunks, Recursive: 11751 chunks

✅ Fixed-size chunks created: 9421
✅ Recursive chunks created: 12228


In [9]:

# ============================================
# Cell 7: SAVE CHUNK FILES (THIS IS CRITICAL!)
# ============================================

print("\n" + "="*60)
print("SAVING CHUNK FILES")
print("="*60)

# Save fixed chunks
with open('chunks_fixed.json', 'w') as f:
    json.dump(all_chunks_fixed, f, indent=2)
print(f"✅ Saved 'chunks_fixed.json' ({len(all_chunks_fixed)} chunks)")

# Save recursive chunks
with open('chunks_recursive.json', 'w') as f:
    json.dump(all_chunks_recursive, f, indent=2)
print(f"✅ Saved 'chunks_recursive.json' ({len(all_chunks_recursive)} chunks)")

# Also save as CSV for easier viewing
try:
    df_fixed = pd.DataFrame(all_chunks_fixed)
    df_fixed.to_csv('chunks_fixed.csv', index=False)
    print(f"✅ Saved 'chunks_fixed.csv'")
    
    df_recursive = pd.DataFrame(all_chunks_recursive)
    df_recursive.to_csv('chunks_recursive.csv', index=False)
    print(f"✅ Saved 'chunks_recursive.csv'")
except:
    print("⚠ Could not save CSV files (pandas issue), but JSON files are saved")

# Verify files exist
print("\n" + "="*60)
print("FILE VERIFICATION")
print("="*60)

required_files = ['chunks_fixed.json', 'chunks_recursive.json', 'pakistan_history_raw.json']
for file in required_files:
    if os.path.exists(file):
        size = os.path.getsize(file) / 1024  # Size in KB
        print(f"✅ {file} exists ({size:.1f} KB)")
    else:
        print(f"❌ {file} MISSING - Something went wrong!")

# Display sample
print("\n📝 Sample chunk from fixed strategy:")
if all_chunks_fixed:
    print(f"  Source: {all_chunks_fixed[0]['source_title']}")
    print(f"  Text: {all_chunks_fixed[0]['text'][:200]}...")
    
print("\n📝 Sample chunk from recursive strategy:")
if all_chunks_recursive:
    print(f"  Source: {all_chunks_recursive[0]['source_title']}")
    print(f"  Text: {all_chunks_recursive[0]['text'][:200]}...")

print("\n" + "="*60)
print("✅ PHASE 1 COMPLETE!")
print("="*60)
print("\n📋 Summary:")
print(f"   - Documents scraped: {len(all_documents)}")
print(f"   - Fixed chunks: {len(all_chunks_fixed)}")
print(f"   - Recursive chunks: {len(all_chunks_recursive)}")
print("\n➡️ You can now run Phase 2 (upload_to_pinecone)")


SAVING CHUNK FILES
✅ Saved 'chunks_fixed.json' (9421 chunks)
✅ Saved 'chunks_recursive.json' (12228 chunks)
✅ Saved 'chunks_fixed.csv'
✅ Saved 'chunks_recursive.csv'

FILE VERIFICATION
✅ chunks_fixed.json exists (6625.2 KB)
✅ chunks_recursive.json exists (6808.4 KB)
✅ pakistan_history_raw.json exists (4415.4 KB)

📝 Sample chunk from fixed strategy:
  Source: Indus Valley Civilisation
  Text: The Indus Valley Civilisation (IVC), also known as the Indus Civilisation or the Harappan Civilisation, was a Bronze Age civilisation in the northwestern regions of South Asia, lasting from 3300 BCE t...

📝 Sample chunk from recursive strategy:
  Source: Indus Valley Civilisation
  Text: The Indus Valley Civilisation (IVC), also known as the Indus Civilisation or the Harappan Civilisation, was a Bronze Age civilisation in the northwestern regions of South Asia, lasting from 3300 BCE t...

✅ PHASE 1 COMPLETE!

📋 Summary:
   - Documents scraped: 107
   - Fixed chunks: 9421
   - Recursive chunks: 122